# Classification — Non-Temporal Approach

This notebook builds and evaluates a non-temporal classification pipeline for **next-day mood prediction**.
The engineered feature dataset was produced by `1c_feature_engineering_non_temporal/feature_engineering.ipynb`.

## Pipeline overview
1. Load train / val / test splits (window size = 7 days)
2. Define the 3-class target (tertile thresholds from train only)
3. Preprocessing: median imputation
4. Baseline models: Dummy classifier + Logistic Regression
5. Main model: Random Forest with randomised hyperparameter search
6. Secondary model: Histogram Gradient Boosting (handles NaN natively)
7. Compare all models on the validation set; select best
8. Retrain best model on train + val; evaluate on held-out test set
9. Report: confusion matrix, per-class metrics, feature importances

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

DATA_DIR = Path("../1c_feature_engineering_non_temporal/output")

---
## 1. Load data

In [ ]:
train_raw = pd.read_parquet(DATA_DIR / "train_w7.parquet")
val_raw = pd.read_parquet(DATA_DIR / "val_w7.parquet")
test_raw = pd.read_parquet(DATA_DIR / "test_w7.parquet")

META_COLS = ["id", "target_date", "target_mood"]
FEAT_COLS = [c for c in train_raw.columns if c not in META_COLS]

print(f"Train : {len(train_raw):>4} instances | {len(FEAT_COLS)} features")
print(f"Val   : {len(val_raw):>4} instances")
print(f"Test  : {len(test_raw):>4} instances")
print(f"Patients: {pd.concat([train_raw, val_raw, test_raw])['id'].nunique()}")
print("\ntarget_mood (train):")
print(train_raw["target_mood"].describe().round(3).to_string())
print(f"\nTrain feature missingness: {train_raw[FEAT_COLS].isna().mean().mean():.1%}")
print(f"Val   feature missingness: {val_raw[FEAT_COLS].isna().mean().mean():.1%}")
print(f"Test  feature missingness: {test_raw[FEAT_COLS].isna().mean().mean():.1%}")

---
## 2. Classification target: 3-class mood

The raw target `target_mood` is a continuous value (mean daily mood on a 1–10 scale).
To frame this as a classification problem we discretise it into three ordered classes.

### Choice: tertile thresholds from the training set

We split the training-set target distribution at the 33rd and 67th percentiles:

| Class | Label | Condition |
|---|---|---|
| 0 | **Low** | mood ≤ train 33rd percentile |
| 1 | **Medium** | train 33rd < mood ≤ train 67th percentile |
| 2 | **High** | mood > train 67th percentile |

**Rationale:**
- Thresholds computed on the *training set only* to prevent leakage into val/test.
- Tertile split maximises class balance, which matters for macro F1.
- Three classes give a clinically meaningful distinction (low / medium / high wellbeing).
- Fixed absolute thresholds (e.g. ≤5 / ≤7 / >7) were rejected because they produce
  very imbalanced classes on this dataset (>540 instances in the "high" bin vs <130 in "low").

In [ ]:
# Thresholds computed on the training set ONLY
Q33 = train_raw["target_mood"].quantile(1 / 3)
Q67 = train_raw["target_mood"].quantile(2 / 3)
CLASS_NAMES = ["Low", "Medium", "High"]

print(f"Thresholds (from train):  low < {Q33:.4f}  <=  mid  < {Q67:.4f}  <= high")


def mood_to_class(series: pd.Series, q33: float, q67: float) -> np.ndarray:
    """Discretise continuous mood into 0/1/2 classes using pre-computed thresholds."""
    return (
        pd.cut(
            series,
            bins=[-np.inf, q33, q67, np.inf],
            labels=[0, 1, 2],
        )
        .astype(int)
        .values
    )


y_train = mood_to_class(train_raw["target_mood"], Q33, Q67)
y_val = mood_to_class(val_raw["target_mood"], Q33, Q67)
y_test = mood_to_class(test_raw["target_mood"], Q33, Q67)

for split_name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    vals, counts = np.unique(y, return_counts=True)
    dist = {CLASS_NAMES[v]: c for v, c in zip(vals, counts)}
    print(f"{split_name:5s}: {dist}  (total {len(y)})")

---
## 3. Preprocessing

**Imputation:** ~7% of feature values are missing (mostly lag features and same-weekday lags
early in a patient's observation period, and sparse app-category features).
We use **median imputation** fit on the training set only.

The missingness itself is already captured by dedicated `_n_missing_days` and `_obs_frac`
features in the dataset, so imputing the value does not hide the absence signal.

**Scaling:** applied in the Logistic Regression pipeline only.
Tree models (RF, HistGB) are scale-invariant.

In [ ]:
X_train_raw = train_raw[FEAT_COLS].values
X_val_raw = val_raw[FEAT_COLS].values
X_test_raw = test_raw[FEAT_COLS].values

# Fit imputer on train only
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train_raw)
X_val = imputer.transform(X_val_raw)
X_test = imputer.transform(X_test_raw)

print(f"Post-imputation NaN in train : {np.isnan(X_train).sum()}")
print(f"Post-imputation NaN in val   : {np.isnan(X_val).sum()}")
print(f"Post-imputation NaN in test  : {np.isnan(X_test).sum()}")

# For models that use the predefined split (tune on val),
# stack train + val into one array
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

# PredefinedSplit: -1 = always in training fold, 0 = validation fold
split_idx = [-1] * len(X_train) + [0] * len(X_val)
pre_split = PredefinedSplit(test_fold=split_idx)

# HistGB uses the raw arrays (handles NaN natively)
X_train_raw_hgb = X_train_raw
X_val_raw_hgb = X_val_raw
X_test_raw_hgb = X_test_raw
X_trainval_raw_hgb = np.vstack([X_train_raw_hgb, X_val_raw_hgb])

print("\nPreprocessing complete.")

---
## 4. Evaluation setup

### Why we do NOT use random cross-validation

The dataset contains **repeated observations from 27 patients**. Random k-fold would mix
past and future observations from the same patient across folds, causing temporal leakage.

### Our approach

We use the **pre-existing time-wise per-patient splits** produced by the feature engineering
step. For each patient the first 60% of their target dates go to train, the next 20%
to val, and the last 20% to test.

- **Hyperparameter tuning**: `PredefinedSplit` trains on the train fold and scores on val.
- **Model selection**: best macro F1 on val.
- **Final reporting**: best model re-trained on train + val, evaluated on test.

### Primary metric: macro F1

Macro F1 averages F1 equally across the three classes, regardless of support.
This is appropriate because:
- We care equally about predicting low, medium, and high mood correctly.
- There is mild class imbalance (Medium class is ~20% smaller than the others).
- In a wellbeing context, missing a "low mood" day is just as costly as missing a "high mood" day.

In [ ]:
def eval_on_val(model, X_v=X_val, y_v=y_val, label="") -> dict:
    """Fit model on train, evaluate on val. Returns metric dict."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_v)
    acc = accuracy_score(y_v, y_pred)
    f1_mac = f1_score(y_v, y_pred, average="macro")
    f1_wt = f1_score(y_v, y_pred, average="weighted")
    if label:
        print(
            f"{label:40s}  acc={acc:.3f}  macro-F1={f1_mac:.3f}  weighted-F1={f1_wt:.3f}"
        )
    return {"model": label, "acc": acc, "macro_f1": f1_mac, "weighted_f1": f1_wt}


def full_report(model, X_tr, y_tr, X_te, y_te, label=""):
    """Fit on X_tr/y_tr, full report on X_te/y_te."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(f"\n{'=' * 60}")
    print(f" {label}")
    print(f"{'=' * 60}")
    print(f"Accuracy        : {accuracy_score(y_te, y_pred):.4f}")
    print(f"Macro F1        : {f1_score(y_te, y_pred, average='macro'):.4f}")
    print(f"Weighted F1     : {f1_score(y_te, y_pred, average='weighted'):.4f}")
    print("\nPer-class report:")
    print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))
    return y_pred

---
## 5. Baseline models

### 5a. Dummy classifier (majority-class)

Always predicts the most frequent class in the training set.
This is the weakest possible baseline — any real model must beat it.
It establishes the floor for accuracy and exposes how much is due to class imbalance.

### 5b. Logistic Regression

A linear classifier with L2 regularisation.
- Interpretable coefficients.
- Fast to train; no tree structure to overfit on small datasets.
- Represents what can be achieved with a *linear* combination of the engineered features.
- Requires imputation + scaling (included in a Pipeline).

In [ ]:
results = []  # collect all model results

# --- 5a. Dummy classifier ---
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
results.append(eval_on_val(dummy, label="Dummy (majority class)"))

# --- 5b. Logistic Regression ---
# Pipeline bundles its own imputer + scaler so it receives raw arrays directly.
lr_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                C=0.1,  # moderate regularisation
            ),
        ),
    ]
)
lr_pipe.fit(X_train_raw, y_train)
lr_pred_val = lr_pipe.predict(X_val_raw)
lr_result = {
    "model": "Logistic Regression (C=0.1, balanced)",
    "acc": accuracy_score(y_val, lr_pred_val),
    "macro_f1": f1_score(y_val, lr_pred_val, average="macro"),
    "weighted_f1": f1_score(y_val, lr_pred_val, average="weighted"),
}
results.append(lr_result)
print(
    f"{'Logistic Regression (C=0.1, balanced)':40s}  "
    f"acc={lr_result['acc']:.3f}  "
    f"macro-F1={lr_result['macro_f1']:.3f}  "
    f"weighted-F1={lr_result['weighted_f1']:.3f}"
)

---
## 6. Main model: Random Forest

**Why Random Forest?**
- Captures non-linear interactions between features (e.g. mood lag × social app usage).
- Robust to the moderate feature dimensionality (136 features, ~740 training instances).
- Handles mixed feature types without scaling.
- Provides reliable feature importances for interpretation.
- Less sensitive to outliers than Gradient Boosting; good default choice for tabular data of this size.

### Hyperparameter search

We use `RandomizedSearchCV` with `PredefinedSplit` (train → fitting, val → scoring),
scored by macro F1. 80 random draws from the search space.

| Parameter | Candidates | Rationale |
|---|---|---|
| `n_estimators` | 100, 200, 300, 500 | More trees → lower variance; 500 is a practical ceiling |
| `max_depth` | None, 5, 10, 15, 20 | Limits overfitting |
| `min_samples_split` | 2, 5, 10 | Controls tree growth |
| `min_samples_leaf` | 1, 2, 4 | Leaf smoothing |
| `max_features` | 'sqrt', 'log2', 0.3, 0.5 | Feature subsampling per split |
| `class_weight` | 'balanced', None | Handles mild class imbalance |

In [ ]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "class_weight": ["balanced", None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=rf_param_dist,
    n_iter=80,
    cv=pre_split,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=False,  # we will retrain manually
    verbose=1,
)

print("Tuning Random Forest (80 iterations, PredefinedSplit)...")
rf_search.fit(X_trainval, y_trainval)

best_rf_params = rf_search.best_params_
best_rf_val_f1 = rf_search.best_score_
print(f"\nBest val macro-F1 : {best_rf_val_f1:.4f}")
print(f"Best params       : {best_rf_params}")

In [ ]:
# Evaluate the best RF config on val (trained on train only)
best_rf = RandomForestClassifier(
    **best_rf_params,
    random_state=RANDOM_STATE,
)
results.append(eval_on_val(best_rf, label="Random Forest (tuned)"))

---
## 7. Secondary model: Histogram Gradient Boosting

`HistGradientBoostingClassifier` (sklearn's efficient gradient boosting implementation):
- Handles NaN values natively — no imputation step needed.
- Typically outperforms Random Forest on tabular data when data is sufficient.
- Acts as a sanity check: if it matches or beats RF, gradient boosting is a strong choice.

| Parameter | Candidates |
|---|---|
| `max_iter` | 100, 200, 300 |
| `max_depth` | 3, 4, 5, 6 |
| `learning_rate` | 0.01, 0.05, 0.1, 0.2 |
| `min_samples_leaf` | 10, 20, 30 |
| `l2_regularization` | 0.0, 0.1, 1.0 |

In [ ]:
hgb_param_dist = {
    "max_iter": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "min_samples_leaf": [10, 20, 30],
    "l2_regularization": [0.0, 0.1, 1.0],
    "class_weight": ["balanced", None],
}

hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(
        random_state=RANDOM_STATE, categorical_features=None
    ),
    param_distributions=hgb_param_dist,
    n_iter=80,
    cv=pre_split,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=False,
    verbose=1,
)

print("Tuning HistGradientBoosting (80 iterations, PredefinedSplit)...")
hgb_search.fit(X_trainval_raw_hgb, y_trainval)

best_hgb_params = hgb_search.best_params_
best_hgb_val_f1 = hgb_search.best_score_
print(f"\nBest val macro-F1 : {best_hgb_val_f1:.4f}")
print(f"Best params       : {best_hgb_params}")

In [ ]:
# Evaluate best HGB config on val
best_hgb = HistGradientBoostingClassifier(
    **best_hgb_params,
    random_state=RANDOM_STATE,
    categorical_features=None,
)
best_hgb.fit(X_train_raw_hgb, y_train)
hgb_pred_val = best_hgb.predict(X_val_raw_hgb)
hgb_acc = accuracy_score(y_val, hgb_pred_val)
hgb_f1m = f1_score(y_val, hgb_pred_val, average="macro")
hgb_f1w = f1_score(y_val, hgb_pred_val, average="weighted")
results.append(
    {
        "model": "Hist. Gradient Boosting (tuned)",
        "acc": hgb_acc,
        "macro_f1": hgb_f1m,
        "weighted_f1": hgb_f1w,
    }
)
print(
    f"{'Hist. Gradient Boosting (tuned)':40s}  acc={hgb_acc:.3f}  macro-F1={hgb_f1m:.3f}  weighted-F1={hgb_f1w:.3f}"
)

---
## 8. MLP (Multilayer Perceptron)

A standard feedforward neural network — non-temporal because it takes the same flat
136-feature vector as input. No recurrence, no attention, no sequence ordering.

```
Input (136) → Linear → BatchNorm → ReLU → Dropout
            → Linear → BatchNorm → ReLU → Dropout
            → Linear → BatchNorm → ReLU → Dropout
            → Linear → Softmax → 3 classes
```

### Design choices
- **BatchNorm** before each activation: stabilises training on small datasets.
- **Dropout** (tuned): main regularisation mechanism against overfitting.
- **Class weights** in CrossEntropyLoss: same rationale as `class_weight='balanced'` in sklearn.
- **Early stopping** on validation macro F1 (patience = 30 epochs): avoids overfitting
  without needing a fixed epoch count.
- **StandardScaler** fitted on train only: MLPs are sensitive to feature scale,
  unlike tree models.

### Hyperparameter search
We try a small grid of architectures and dropout rates, training each for up to 300 epochs
with early stopping, and select the configuration with the best validation macro F1.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score as sk_f1

torch.manual_seed(RANDOM_STATE)

# ── Scaling (MLPs require normalised inputs, unlike tree models) ───────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit on train only
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)


# ── Convert to tensors ─────────────────────────────────────────────────────────
def to_tensors(X, y):
    return (
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )


Xt, yt = to_tensors(X_train_sc, y_train)
Xv, yv = to_tensors(X_val_sc, y_val)
Xs, ys = to_tensors(X_test_sc, y_test)

# ── Class weights for imbalanced classes ──────────────────────────────────────
counts = torch.bincount(yt)
cw = 1.0 / counts.float()
cw = cw / cw.sum() * len(counts)  # normalise so weights sum to n_classes


# ── MLP architecture ──────────────────────────────────────────────────────────
class MoodMLP(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dims: list, dropout: float, n_classes: int = 3
    ):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(
    hidden_dims,
    dropout,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=32,
    max_epochs=300,
    patience=30,
    verbose=False,
):
    """Train one MLP config; return (model, best_val_macro_f1)."""
    model = MoodMLP(Xt.shape[1], hidden_dims, dropout)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    best_f1, best_state, no_improve = 0.0, None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            optimiser.zero_grad()
            criterion(model(xb), yb).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            preds = model(Xv).argmax(dim=1).numpy()
        val_f1 = sk_f1(yv.numpy(), preds, average="macro")

        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, model.state_dict(), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"  Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return model, best_f1


# ── Hyperparameter grid ───────────────────────────────────────────────────────
mlp_configs = [
    {"hidden_dims": [128, 64, 32], "dropout": 0.3},
    {"hidden_dims": [128, 64, 32], "dropout": 0.5},
    {"hidden_dims": [256, 128, 64], "dropout": 0.3},
    {"hidden_dims": [256, 128, 64], "dropout": 0.5},
    {"hidden_dims": [64, 32], "dropout": 0.3},
    {"hidden_dims": [64, 32], "dropout": 0.5},
]

print("Training MLP configurations...")
mlp_results = []
for cfg in mlp_configs:
    model, val_f1 = train_mlp(**cfg)
    label = f"MLP {cfg['hidden_dims']} drop={cfg['dropout']}"
    print(f"  {label:40s}  val macro-F1={val_f1:.4f}")
    mlp_results.append((val_f1, model, label))

# Best config by val macro F1
mlp_results.sort(key=lambda x: x[0], reverse=True)
best_mlp_val_f1, best_mlp_model, best_mlp_label = mlp_results[0]
print(f"\nBest MLP: {best_mlp_label}  →  val macro-F1={best_mlp_val_f1:.4f}")

In [ ]:
# Add best MLP val result to the results list
best_mlp_model.eval()
with torch.no_grad():
    mlp_val_preds = best_mlp_model(Xv).argmax(dim=1).numpy()

results.append(
    {
        "model": f"MLP (best: {best_mlp_label})",
        "acc": accuracy_score(y_val, mlp_val_preds),
        "macro_f1": sk_f1(y_val, mlp_val_preds, average="macro"),
        "weighted_f1": sk_f1(y_val, mlp_val_preds, average="weighted"),
    }
)
r = results[-1]
print(
    f"{r['model']:50s}  acc={r['acc']:.3f}  macro-F1={r['macro_f1']:.3f}  weighted-F1={r['weighted_f1']:.3f}"
)

---
## 8. Validation set comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df = results_df.rename(
    columns={
        "model": "Model",
        "acc": "Accuracy",
        "macro_f1": "Macro F1",
        "weighted_f1": "Weighted F1",
    }
).reset_index(drop=True)
results_df[["Accuracy", "Macro F1", "Weighted F1"]] = results_df[
    ["Accuracy", "Macro F1", "Weighted F1"]
].round(4)
print("=== Validation set results ===")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
print(f"\n>>> Best model on val: {best_model_name}")

---
## 9. Final evaluation on test set

The best model (highest macro F1 on val) is retrained on **train + val combined**,
then evaluated once on the held-out test set.

Retraining on train+val before the final test evaluation is standard practice:
it uses all available labelled data for the final model without touching the test set
during any part of the selection process.

In [ ]:
# --- Baseline final test report ---
print("BASELINE: Logistic Regression")
lr_pipe.fit(np.vstack([X_train_raw, X_val_raw]), y_trainval)
lr_pred_test_final = lr_pipe.predict(X_test_raw)
print(f"  Test accuracy  : {accuracy_score(y_test, lr_pred_test_final):.4f}")
print(f"  Test macro F1  : {f1_score(y_test, lr_pred_test_final, average='macro'):.4f}")
print(classification_report(y_test, lr_pred_test_final, target_names=CLASS_NAMES))

# --- Random Forest: retrain on train+val, evaluate on test ---
print("\nRANDOM FOREST (retrained on train+val)")
best_rf_final = RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE)
y_test_pred = full_report(
    best_rf_final,
    X_tr=X_trainval,
    y_tr=y_trainval,
    X_te=X_test,
    y_te=y_test,
    label="Random Forest — test set",
)

# --- HistGB: retrain on train+val, evaluate on test ---
print("\nHIST. GRADIENT BOOSTING (retrained on train+val)")
best_hgb_final = HistGradientBoostingClassifier(
    **best_hgb_params, random_state=RANDOM_STATE, categorical_features=None
)
y_test_pred_hgb = full_report(
    best_hgb_final,
    X_tr=X_trainval_raw_hgb,
    y_tr=y_trainval,
    X_te=X_test_raw_hgb,
    y_te=y_test,
    label="Hist. Gradient Boosting — test set",
)

# --- MLP: retrain on train+val, evaluate on test ---
print("\nMLP (retrained on train+val)")

# Scale train+val together, then scale test
scaler_final = StandardScaler()
X_trainval_sc_final = scaler_final.fit_transform(
    np.vstack([X_train_sc, X_val_sc])  # already individually scaled — redo from raw
)
# Redo from the imputed (but unscaled) arrays
scaler_final2 = StandardScaler()
X_trainval_sc2 = scaler_final2.fit_transform(np.vstack([X_train, X_val]))
X_test_sc2 = scaler_final2.transform(X_test)

Xt_final = torch.tensor(X_trainval_sc2, dtype=torch.float32)
yt_final = torch.tensor(y_trainval, dtype=torch.long)
Xs_final = torch.tensor(X_test_sc2, dtype=torch.float32)

# Reparse best config from label: retrain best architecture on train+val
best_cfg_idx = mlp_results[0][0]  # val f1 (not used, just to confirm)
best_cfg = mlp_configs[
    [f"MLP {c['hidden_dims']} drop={c['dropout']}" for c in mlp_configs].index(
        best_mlp_label
    )
]

# Counts and class weights on full trainval set
cw_final = 1.0 / torch.bincount(yt_final).float()
cw_final = cw_final / cw_final.sum() * 3

mlp_final = MoodMLP(Xt_final.shape[1], best_cfg["hidden_dims"], best_cfg["dropout"])
criterion_f = nn.CrossEntropyLoss(weight=cw_final)
opt_f = torch.optim.Adam(mlp_final.parameters(), lr=1e-3, weight_decay=1e-4)
loader_f = DataLoader(TensorDataset(Xt_final, yt_final), batch_size=32, shuffle=True)

torch.manual_seed(RANDOM_STATE)
for epoch in range(200):
    mlp_final.train()
    for xb, yb in loader_f:
        opt_f.zero_grad()
        criterion_f(mlp_final(xb), yb).backward()
        opt_f.step()

mlp_final.eval()
with torch.no_grad():
    y_test_pred_mlp = mlp_final(Xs_final).argmax(dim=1).numpy()

print(f"\n{'=' * 60}")
print(" MLP — test set (retrained on train+val)")
print(f"{'=' * 60}")
print(f"Accuracy        : {accuracy_score(y_test, y_test_pred_mlp):.4f}")
print(f"Macro F1        : {f1_score(y_test, y_test_pred_mlp, average='macro'):.4f}")
print(f"Weighted F1     : {f1_score(y_test, y_test_pred_mlp, average='weighted'):.4f}")
print("\nPer-class report:")
print(classification_report(y_test, y_test_pred_mlp, target_names=CLASS_NAMES))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (y_pred_plot, title) in zip(
    axes,
    [
        (lr_pred_test_final, "Logistic Regression\n(baseline)"),
        (y_test_pred, "Random Forest\n(main)"),
        (y_test_pred_hgb, "Hist. Gradient\nBoosting"),
        (y_test_pred_mlp, "MLP"),
    ],
):
    cm = confusion_matrix(y_test, y_pred_plot)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
        cbar=False,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

plt.suptitle("Confusion matrices — test set", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices.png")

---
## 10. Feature importances (Random Forest)

In [ ]:
importances = best_rf_final.feature_importances_
feat_imp_df = (
    pd.DataFrame({"feature": FEAT_COLS, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 most important features:")
print(feat_imp_df.head(20).to_string(index=False))

# Plot top 25
top_n = 25
fig, ax = plt.subplots(figsize=(8, 8))
feat_imp_df.head(top_n).sort_values("importance").plot(
    kind="barh", x="feature", y="importance", ax=ax, legend=False, color="steelblue"
)
ax.set_xlabel("Mean decrease in impurity")
ax.set_title(f"Top {top_n} feature importances — Random Forest")
plt.tight_layout()
plt.savefig("feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_importances.png")

---
## 11. Final results summary

In [ ]:
test_results = []
dummy_final = DummyClassifier(strategy="most_frequent").fit(X_trainval, y_trainval)
for name, y_pred_t in [
    ("Dummy (majority class)", dummy_final.predict(X_test)),
    ("Logistic Regression (baseline)", lr_pred_test_final),
    ("Random Forest (tuned)", y_test_pred),
    ("Hist. Gradient Boosting (tuned)", y_test_pred_hgb),
    ("MLP (tuned)", y_test_pred_mlp),
]:
    test_results.append(
        {
            "Model": name,
            "Accuracy": round(accuracy_score(y_test, y_pred_t), 4),
            "Macro F1": round(f1_score(y_test, y_pred_t, average="macro"), 4),
            "Weighted F1": round(f1_score(y_test, y_pred_t, average="weighted"), 4),
        }
    )

test_df = (
    pd.DataFrame(test_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)
print("=== Final test set results ===")
print(test_df.to_string(index=False))

---
## 12. Discussion

### Classification target
Daily mean mood was discretised into three balanced classes (Low / Medium / High) using
tertile thresholds computed from the training set only (Low ≤ 6.80, Medium ≤ 7.33, High > 7.33).
This avoids leakage and produces roughly equal class sizes, making macro F1 the appropriate
primary metric.

### Baseline vs. main model
The dummy classifier establishes a lower bound by always predicting the majority class.
Logistic Regression shows the benefit of a linear model over the engineered features.
The Random Forest and HistGradientBoosting capture non-linear interactions — for example,
the interaction between recent mood lags and behavioural features like social app usage.

### Feature importance interpretation
The most important features are consistently the **recent mood lags** (`mood_lag1`,
`mood_lag2`, `mood_win_mean`), confirming that next-day mood is strongly autocorrelated.
Behavioural features (screen time, social app usage) and missingness indicators
contribute additional signal beyond the mood history alone.

### Limitations
- The dataset is small (744 training instances, 27 patients). Performance estimates
  have wide confidence intervals.
- Patient-level effects are not modelled; a mixed-effects or patient-stratified model
  may perform better.
- The 7-day window was chosen as the default; the window-size comparison from the
  feature engineering notebook should guide final window selection.

### Reproducibility
All random operations use `RANDOM_STATE = 42`. The preprocessing pipeline (imputer fitted
on train only) and the PredefinedSplit hyperparameter search are fully deterministic.
Run the cells top-to-bottom to reproduce all results.